# BERTopic Analysis\nAnalysis of topic distributions from `bertopic_avg_topics.csv` and `bertopic_avg_listing_topics.csv`.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

DATA_DIR = Path("data")

topics_df = pd.read_csv(DATA_DIR / "bertopic_avg_topics.csv")
listings_df = pd.read_csv(DATA_DIR / "bertopic_avg_listing_topics.csv")

print(f"Topics: {len(topics_df)}")
print(f"Listings: {len(listings_df):,}")
print(f"\nTopics columns: {list(topics_df.columns)}")
print(f"Listings columns: {list(listings_df.columns)}")

## 1. Topic Overview

In [ ]:
# Display all topics with their top words and counts
topics_display = topics_df[topics_df["Topic"] != -1].copy()
topics_display

## 2. Topic Size Distribution

In [ ]:
# Bar chart of topic sizes (excluding outlier topic -1)
fig = px.bar(
    topics_display.sort_values("Count", ascending=True),
    x="Count", y="Name", orientation="h",
    title="Number of Listings per Topic",
    labels={"Name": "Topic", "Count": "Number of Listings"},
)
fig.update_layout(height=max(400, len(topics_display) * 30), yaxis_tickfont_size=10)
fig.show()

# Outlier stats
outlier_row = topics_df[topics_df["Topic"] == -1]
if not outlier_row.empty:
    outlier_count = outlier_row["Count"].values[0]
    total = topics_df["Count"].sum()
    print(f"Outlier listings (Topic -1): {outlier_count:,} ({outlier_count/total*100:.1f}%)")

## 3. Dominant Topic Distribution Across Listings

In [ ]:
# Histogram of dominant topics
topic_counts = listings_df["dominant_topic"].value_counts().sort_index().reset_index()
topic_counts.columns = ["topic", "count"]
topic_counts = topic_counts[topic_counts["topic"] != -1]

fig = px.bar(
    topic_counts, x="topic", y="count",
    title="Dominant Topic Assignment Across Listings",
    labels={"topic": "Topic ID", "count": "Number of Listings"},
)
fig.update_layout(xaxis=dict(dtick=1))
fig.show()

## 4. Topic Probability Heatmap (sample)

In [ ]:
# Heatmap of topic probabilities for a random sample of listings
prob_cols = [c for c in listings_df.columns if c.endswith("_prob") and c.startswith("topic_")]
sample = listings_df[listings_df["dominant_topic"] != -1].sample(n=min(50, len(listings_df)), random_state=42)

fig = px.imshow(
    sample[prob_cols].values,
    x=[c.replace("_prob", "") for c in prob_cols],
    y=sample["listing_id"].astype(str),
    color_continuous_scale="Blues",
    title="Topic Probability Distribution (50 sample listings)",
    labels={"x": "Topic", "y": "Listing ID", "color": "Probability"},
    aspect="auto",
)
fig.update_layout(height=800, yaxis_tickfont_size=8)
fig.show()

## 5. Average Topic Probabilities Across All Listings

In [ ]:
# Average probability per topic across all listings
mean_probs = listings_df[prob_cols].mean().reset_index()
mean_probs.columns = ["topic", "avg_probability"]
mean_probs["topic"] = mean_probs["topic"].str.replace("_prob", "")

fig = px.bar(
    mean_probs.sort_values("avg_probability", ascending=True),
    x="avg_probability", y="topic", orientation="h",
    title="Average Topic Probability Across All Listings",
    labels={"avg_probability": "Mean Probability", "topic": "Topic"},
)
fig.show()

## 6. Topic Confidence: How Confident Are Assignments?

In [ ]:
# Distribution of top-1 topic probability (confidence)
fig = px.histogram(
    listings_df[listings_df["dominant_topic"] != -1],
    x="top1_prob", nbins=50,
    title="Distribution of Dominant Topic Probability (Confidence)",
    labels={"top1_prob": "Top-1 Topic Probability", "count": "Number of Listings"},
)
fig.add_vline(x=listings_df["top1_prob"].median(), line_dash="dash",
              annotation_text=f"Median: {listings_df['top1_prob'].median():.3f}")
fig.show()

# Stats
print(f"Mean confidence:   {listings_df['top1_prob'].mean():.3f}")
print(f"Median confidence: {listings_df['top1_prob'].median():.3f}")
print(f"Min confidence:    {listings_df['top1_prob'].min():.3f}")
print(f"Max confidence:    {listings_df['top1_prob'].max():.3f}")

## 7. Topic Correlation Matrix

In [ ]:
# Correlation between topic probabilities — which topics co-occur?
corr = listings_df[prob_cols].corr()
labels = [c.replace("_prob", "") for c in prob_cols]

fig = px.imshow(
    corr.values, x=labels, y=labels,
    color_continuous_scale="RdBu_r", zmin=-1, zmax=1,
    title="Topic Probability Correlation Matrix",
    aspect="equal",
)
fig.update_layout(height=600, width=700)
fig.show()

## 8. Top-3 Topic Breakdown per Listing

In [ ]:
# How much of the probability mass is captured by the top 3 topics?
listings_non_outlier = listings_df[listings_df["dominant_topic"] != -1].copy()

top3_cols = [c for c in ["top1_prob", "top2_prob", "top3_prob"] if c in listings_df.columns]
listings_non_outlier["top3_total"] = listings_non_outlier[top3_cols].sum(axis=1)

fig = px.histogram(
    listings_non_outlier, x="top3_total", nbins=50,
    title="Top-3 Topics: Cumulative Probability Coverage",
    labels={"top3_total": "Sum of Top-3 Topic Probabilities"},
)
fig.add_vline(x=listings_non_outlier["top3_total"].median(), line_dash="dash",
              annotation_text=f"Median: {listings_non_outlier['top3_total'].median():.3f}")
fig.show()

print(f"Listings where top-3 topics cover >80% probability: "
      f"{(listings_non_outlier['top3_total'] > 0.8).sum():,} "
      f"({(listings_non_outlier['top3_total'] > 0.8).mean()*100:.1f}%)")

## 9. Topic Diversity: How Many Topics Does Each Listing Touch?

In [ ]:
# Count topics with >5% probability per listing (topic diversity)
threshold = 0.05
listings_non_outlier["n_topics_above_5pct"] = (
    listings_non_outlier[prob_cols] > threshold
).sum(axis=1)

fig = px.histogram(
    listings_non_outlier, x="n_topics_above_5pct",
    title=f"Number of Topics with >{threshold*100:.0f}% Probability per Listing",
    labels={"n_topics_above_5pct": "Number of Topics", "count": "Listings"},
)
fig.update_layout(xaxis=dict(dtick=1))
fig.show()

print(listings_non_outlier["n_topics_above_5pct"].describe())

## 10. Join with Feature Matrix

In [ ]:
# Merge topic data with feature matrix for downstream analysis
features_df = pd.read_csv(DATA_DIR / "feature_matrix.csv")

# listing_id in topics, id in feature matrix
merged = features_df.merge(
    listings_df, left_on="id", right_on="listing_id", how="inner"
)
print(f"Feature matrix:  {len(features_df):,} listings")
print(f"Topic data:      {len(listings_df):,} listings")
print(f"Merged:          {len(merged):,} listings")
merged.head()

## 11. Topic vs. Price

In [ ]:
# Price distribution by dominant topic
merged_valid = merged[merged["dominant_topic"] != -1].copy()
merged_valid["dominant_topic"] = merged_valid["dominant_topic"].astype(str)

fig = px.box(
    merged_valid, x="dominant_topic", y="price_numeric",
    title="Price Distribution by Dominant Topic",
    labels={"dominant_topic": "Topic", "price_numeric": "Price ($)"},
)
fig.update_layout(xaxis=dict(dtick=1))
fig.show()

# Median price per topic
print("\nMedian price per topic:")
print(merged_valid.groupby("dominant_topic")["price_numeric"].median().sort_values(ascending=False))

## 12. Topic vs. Review Scores

In [ ]:
# Average review scores per dominant topic
score_cols = [c for c in merged.columns if c.startswith("review_scores_")]
topic_scores = merged_valid.groupby("dominant_topic")[score_cols].mean()

fig = px.imshow(
    topic_scores.values,
    x=[c.replace("review_scores_", "") for c in score_cols],
    y=topic_scores.index.astype(str),
    color_continuous_scale="RdYlGn",
    title="Mean Review Scores by Dominant Topic",
    labels={"x": "Score Category", "y": "Topic", "color": "Score"},
    aspect="auto",
)
fig.update_layout(height=500)
fig.show()

## 13. Topic vs. Borough

In [ ]:
# Topic distribution by borough
boro_cols = [c for c in merged.columns if c.startswith("boro_")]
merged_valid["borough"] = merged_valid[boro_cols].idxmax(axis=1).str.replace("boro_", "")

topic_boro = pd.crosstab(merged_valid["dominant_topic"], merged_valid["borough"], normalize="index")

fig = px.bar(
    topic_boro.reset_index().melt(id_vars="dominant_topic"),
    x="dominant_topic", y="value", color="borough",
    title="Topic Distribution by Borough (normalized)",
    labels={"dominant_topic": "Topic", "value": "Proportion", "borough": "Borough"},
    barmode="stack",
)
fig.update_layout(xaxis=dict(dtick=1))
fig.show()

## 14. Topic vs. Room Type

In [ ]:
# Topic distribution by room type
prop_cols = [c for c in merged.columns if c.startswith("prop_")]
merged_valid["room_type"] = merged_valid[prop_cols].idxmax(axis=1).str.replace("prop_", "")

topic_room = pd.crosstab(merged_valid["dominant_topic"], merged_valid["room_type"], normalize="index")

fig = px.bar(
    topic_room.reset_index().melt(id_vars="dominant_topic"),
    x="dominant_topic", y="value", color="room_type",
    title="Topic Distribution by Room Type (normalized)",
    labels={"dominant_topic": "Topic", "value": "Proportion", "room_type": "Room Type"},
    barmode="stack",
)
fig.update_layout(xaxis=dict(dtick=1))
fig.show()

## 15. Save Merged Dataset

In [ ]:
# Save the merged feature matrix + topic probabilities for modeling
output_path = DATA_DIR / "feature_matrix_with_topics.csv"
merged.to_csv(output_path, index=False)
print(f"Saved merged dataset to {output_path}")
print(f"Shape: {merged.shape}")